# Garbage Classification: Dataset Inspection and Split Plan

This notebook supports the project proposal. It downloads and inspects the actual Kaggle dataset, creates the required visualizations, and records a reproducible **70/15/15 stratified split**. It does **not** train a model.

**Planned validation metric:** macro-averaged F1 score.

## 1. Install and import dependencies

In [ ]:
%pip install -q kagglehub pandas matplotlib seaborn pillow scikit-learn

In [ ]:
from pathlib import Path
import json
import random

import kagglehub
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from PIL import Image, UnidentifiedImageError
from sklearn.model_selection import train_test_split

SEED = 42
DATASET_HANDLE = 'mostafaabla/garbage-classification'
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
EXPECTED_CLASSES = {
    'battery', 'biological', 'brown-glass', 'cardboard', 'clothes',
    'green-glass', 'metal', 'paper', 'plastic', 'shoes', 'trash',
    'white-glass'
}

random.seed(SEED)
sns.set_theme(style='whitegrid')

# In Colab this writes to /content/artifacts. Locally it writes beside the notebook runtime.
OUTPUT_DIR = Path('/content/artifacts') if Path('/content').exists() else Path('artifacts')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'Artifacts will be saved to: {OUTPUT_DIR.resolve()}')

## 2. Download the Kaggle dataset

Kaggle may request authentication in some environments. Do not commit Kaggle credentials to GitHub.

In [ ]:
download_path = Path(kagglehub.dataset_download(DATASET_HANDLE))
print(f'Dataset downloaded to: {download_path}')

## 3. Locate the class folders and inspect every image

The code locates the directory whose immediate child folders correspond to the expected 12 labels. It then opens every image to verify readability and record its dimensions.

In [ ]:
def image_files_in(directory: Path):
    return sorted(
        path for path in directory.rglob('*')
        if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
    )


def find_class_root(dataset_root: Path) -> Path:
    candidates = [dataset_root] + [p for p in dataset_root.rglob('*') if p.is_dir()]
    scored = []
    for candidate in candidates:
        child_names = {p.name for p in candidate.iterdir() if p.is_dir()}
        score = len(child_names & EXPECTED_CLASSES)
        if score:
            scored.append((score, candidate))
    if not scored:
        raise FileNotFoundError('Could not locate the expected class folders.')
    score, best = max(scored, key=lambda item: item[0])
    if score != len(EXPECTED_CLASSES):
        found = sorted(p.name for p in best.iterdir() if p.is_dir())
        raise ValueError(f'Expected 12 class folders but matched {score}. Found: {found}')
    return best


class_root = find_class_root(download_path)
print(f'Class-folder root: {class_root}')
print('Classes:', sorted(p.name for p in class_root.iterdir() if p.is_dir()))

In [ ]:
records = []
corrupt_files = []

for class_dir in sorted(p for p in class_root.iterdir() if p.is_dir()):
    for image_path in image_files_in(class_dir):
        try:
            with Image.open(image_path) as image:
                image.verify()
            with Image.open(image_path) as image:
                width, height = image.size
                mode = image.mode
            records.append({
                'relative_path': image_path.relative_to(class_root).as_posix(),
                'label': class_dir.name,
                'width': width,
                'height': height,
                'mode': mode,
            })
        except (UnidentifiedImageError, OSError, ValueError) as error:
            corrupt_files.append({'path': str(image_path), 'error': str(error)})

df = pd.DataFrame(records).sort_values(['label', 'relative_path']).reset_index(drop=True)
if df.empty:
    raise RuntimeError('No readable images were found.')

df.to_csv(OUTPUT_DIR / 'dataset_inventory.csv', index=False)
print(f'Readable images: {len(df):,}')
print(f'Unreadable images: {len(corrupt_files):,}')
display(df.head())

## 4. Dataset size, labels, and class balance

In [ ]:
class_distribution = (
    df['label'].value_counts()
    .rename_axis('label')
    .reset_index(name='image_count')
    .sort_values('image_count', ascending=False)
)
class_distribution['percentage'] = (
    100 * class_distribution['image_count'] / len(df)
).round(2)
class_distribution.to_csv(OUTPUT_DIR / 'class_distribution.csv', index=False)
display(class_distribution)

largest = class_distribution.iloc[0]
smallest = class_distribution.iloc[-1]
imbalance_ratio = largest['image_count'] / smallest['image_count']
print(f'Largest class: {largest.label} ({largest.image_count:,})')
print(f'Smallest class: {smallest.label} ({smallest.image_count:,})')
print(f'Largest/smallest imbalance ratio: {imbalance_ratio:.2f}:1')

In [ ]:
plt.figure(figsize=(11, 6))
ax = sns.barplot(
    data=class_distribution,
    x='image_count',
    y='label',
    hue='label',
    palette='viridis',
    legend=False,
)
ax.set_title('Garbage Classification Dataset: Images per Class')
ax.set_xlabel('Number of images')
ax.set_ylabel('Class')
for container in ax.containers:
    ax.bar_label(container, padding=3, fmt='%d')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'class_distribution.png', dpi=200, bbox_inches='tight')
plt.show()

## 5. Representative samples

One reproducibly selected image is shown for each class. These are actual downloaded samples, not images copied from the dataset webpage.

In [ ]:
sample_rows = (
    df.groupby('label', group_keys=False)
    .sample(n=1, random_state=SEED)
    .sort_values('label')
)

fig, axes = plt.subplots(3, 4, figsize=(14, 10))
for axis, row in zip(axes.flat, sample_rows.itertuples(index=False)):
    image_path = class_root / row.relative_path
    with Image.open(image_path) as image:
        axis.imshow(image.convert('RGB'))
    axis.set_title(row.label.replace('-', ' ').title())
    axis.axis('off')

fig.suptitle('Representative Dataset Samples', fontsize=16)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'representative_samples.png', dpi=200, bbox_inches='tight')
plt.show()

## 6. Image-dimension inspection

In [ ]:
dimension_summary = df[['width', 'height']].describe().round(2)
dimension_summary.to_csv(OUTPUT_DIR / 'image_dimension_summary.csv')
display(dimension_summary)

dimension_counts = (
    df.groupby(['width', 'height'])
    .size()
    .reset_index(name='image_count')
    .sort_values('image_count', ascending=False)
)
print(f'Unique width-height combinations: {len(dimension_counts):,}')
display(dimension_counts.head(10))

## 7. Record the planned 70/15/15 split

This creates a manifest only; it does not move or copy the image files. The split is stratified so each class keeps approximately the same proportion. The test labels must not be used for model selection later.

In [ ]:
train_df, temporary_df = train_test_split(
    df,
    test_size=0.30,
    random_state=SEED,
    stratify=df['label'],
)
validation_df, test_df = train_test_split(
    temporary_df,
    test_size=0.50,
    random_state=SEED,
    stratify=temporary_df['label'],
)

split_frames = []
for split_name, split_df in [
    ('train', train_df),
    ('validation', validation_df),
    ('test', test_df),
]:
    split_copy = split_df.copy()
    split_copy['split'] = split_name
    split_frames.append(split_copy)

split_manifest = (
    pd.concat(split_frames, ignore_index=True)
    .sort_values(['split', 'label', 'relative_path'])
)
split_manifest.to_csv(OUTPUT_DIR / 'split_manifest.csv', index=False)

split_summary = pd.crosstab(split_manifest['label'], split_manifest['split'])
split_summary.loc['TOTAL'] = split_summary.sum(axis=0)
display(split_summary)

observed_percentages = (100 * split_manifest['split'].value_counts() / len(split_manifest)).round(2)
print('Observed split percentages:')
display(observed_percentages)

## 8. Save a proposal-ready machine-readable summary

In [ ]:
summary = {
    'dataset_handle': DATASET_HANDLE,
    'task_type': 'multiclass image classification',
    'readable_image_count': int(len(df)),
    'unreadable_image_count': int(len(corrupt_files)),
    'class_count': int(df['label'].nunique()),
    'classes': sorted(df['label'].unique().tolist()),
    'class_distribution': {
        row.label: int(row.image_count)
        for row in class_distribution.itertuples(index=False)
    },
    'planned_split': {'train': 0.70, 'validation': 0.15, 'test': 0.15},
    'random_seed': SEED,
    'validation_metric': 'macro-averaged F1 score',
    'license': 'Open Data Commons Open Database License (ODbL) 1.0',
    'access': 'Kaggle; account/API authentication may be required',
}

with (OUTPUT_DIR / 'dataset_summary.json').open('w', encoding='utf-8') as file:
    json.dump(summary, file, indent=2)

print(json.dumps(summary, indent=2))
print(f'\nAll proposal-supporting artifacts were saved to {OUTPUT_DIR.resolve()}')